In [ ]:
# Collections Data Forensics

## Objective

#The business currently reports that recovery has improved by 11% month-on-month.

#The objective of this analysis is to independently reconstruct collection performance from the raw operational data and determine:

#1. Whether the reported 11% improvement is real.
#2. What actually changed in collection performance.
#3. Whether the observed change is driven by operational improvement or changes in the underlying portfolio.
#4. Whether data-quality, attribution, duplication, or reporting issues affect the reported metrics.
#5. What evidence supports the recommended business decision.

#This notebook focuses on data forensics and establishes the foundation for the Golden Dataset and subsequent recovery analysis.

### Analytical principle

#The raw data is not assumed to be a single source of truth. Every important metric will be independently defined and validated before being used for business conclusions.

In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [3]:
import os

os.listdir("/content")

['.config', 'sample_data']

In [13]:
import zipfile
from pathlib import Path

zip_path = "/content/collections_30k_dataset (2).zip"
extract_path = "/content/collections_30k_dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)

Dataset extracted to: /content/collections_30k_dataset


In [5]:
for root, dirs, files in os.walk("/content/collections_30k_dataset"):
    csv_files = [f for f in files if f.endswith(".csv")]
    if csv_files:
        print(root)
        print(csv_files)

In [14]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

BASE_PATH = Path("/content/collections_30k_dataset")

def load_csv(name):
    return pd.read_csv(BASE_PATH / name)

accounts = load_csv("accounts.csv")
borrowers = load_csv("borrowers.csv")
agents = load_csv("agents.csv")
agent_sessions = load_csv("agent_sessions.csv")
campaigns = load_csv("campaigns.csv")
daily_targeting = load_csv("daily_targeting.csv")
calls = load_csv("calls.csv")
call_attempts = load_csv("call_attempts.csv")
call_dispositions = load_csv("call_dispositions.csv")
whatsapp_events = load_csv("whatsapp_events.csv")
sms_events = load_csv("sms_events.csv")
field_visits = load_csv("field_visits.csv")
promises_to_pay = load_csv("promises_to_pay.csv")
payments = load_csv("payments.csv")
vendor_telephony = load_csv("vendor_telephony.csv")
complaints = load_csv("complaints.csv")
account_status_history = load_csv("account_status_history.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [ ]:
## 1. Data Forensics

The purpose of this section is not to describe the datasets again, but to determine whether the raw data can be trusted for calculating collection performance.

The investigation focuses on:

1. Payment duplication
2. Payment attribution
3. Entity resolution
4. Timestamp and timezone consistency
5. Vendor and disposition changes
6. Campaign definition changes
7. Denominator manipulation
8. Historical/status inconsistencies
9. Late or future-dated events
10. The business impact of identified data-quality issues

Each issue will be classified as:

- Fact
- Strong Evidence
- Correlation
- Hypothesis

No business conclusion will be made until the underlying data issue has been investigated.

In [15]:
## 2. Payment Forensics

### Question

Can the payment data be trusted as the source for measuring recovery?

Before calculating recovery, we need to establish the true grain of a payment and determine whether duplicate or conflicting payment records can inflate collections.

SyntaxError: invalid syntax (2464716663.py, line 7)

In [16]:
# Basic payment structure

print("Rows:", len(payments))
print("Columns:", len(payments.columns))

display(payments.head())

print("\nDuplicate complete rows:", payments.duplicated().sum())

print("\nUnique payment IDs:", payments["payment_id"].nunique())
print("Duplicate payment IDs:", payments["payment_id"].duplicated().sum())

print("\nUnique payment references:", payments["payment_reference"].nunique())
print("Duplicate payment references:", payments["payment_reference"].duplicated().sum())

Rows: 25500
Columns: 9


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
0,PAYMENT0000001,ACC0015539,BRW0011363,2026-02-27 01:28:12,TXN0000007457,"22,433.23",FAILED,CARD,VND0000001
1,PAYMENT0000002,ACC0004445,BRW0011306,2026-07-23 20:25:20,TXN0000030016,"23,295.11",FAILED,UPI,VND0000004
2,PAYMENT0000003,ACC0003853,BRW0002549,2026-01-11 21:27:46,TXN0000010466,"118,814.97",FAILED,NACH,VND0000013
3,PAYMENT0000004,ACC0017531,BRW0006618,2026-06-16 02:35:21,TXN0000031781,"145,082.17",REVERSED,CASH,VND0000013
4,PAYMENT0000005,ACC0001770,BRW0011492,2026-03-03 06:08:23,TXN0000059257,"115,148.46",SUCCESS,NETBANKING,VND0000013



Duplicate complete rows: 486

Unique payment IDs: 25000
Duplicate payment IDs: 500

Unique payment references: 20821
Duplicate payment references: 4678


In [ ]:
### 2.1 Duplicate Payment IDs

A payment ID should normally identify a unique payment event. However, repeated IDs may represent:

- exact ingestion duplicates,
- retries,
- updates to the same payment,
- or genuine conflicting records.

We therefore inspect the records associated with repeated payment IDs before defining the deduplication rule.

In [17]:
# Find payment IDs that occur more than once

duplicate_payment_ids = (
    payments[
        payments["payment_id"].duplicated(keep=False)
    ]
    .sort_values("payment_id")
)

print("Rows involved:", len(duplicate_payment_ids))
print(
    "Unique duplicated payment IDs:",
    duplicate_payment_ids["payment_id"].nunique()
)

display(duplicate_payment_ids.head(30))

Rows involved: 1000
Unique duplicated payment IDs: 500


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
55,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,"77,982.73",PENDING,NETBANKING,VND0000007
25035,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,"77,982.73",PENDING,NETBANKING,VND0000007
75,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,"43,712.91",PENDING,UPI,VND0000014
25408,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,"43,712.91",PENDING,UPI,VND0000014
107,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,"102,830.42",SUCCESS,NACH,VND0000003
25481,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,"102,830.42",SUCCESS,NACH,VND0000003
148,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,"96,766.12",SUCCESS,CASH,VND0000013
25099,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,"96,766.12",SUCCESS,CASH,VND0000013
198,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,"81,814.00",SUCCESS,NETBANKING,VND0000012
25330,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,"81,814.00",SUCCESS,NETBANKING,VND0000012


In [ ]:
print(payments.columns.tolist())

['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']


In [ ]:
payments.head(3)

,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
0,PAYMENT0000001,ACC0015539,BRW0011363,2026-02-27 01:28:12,TXN0000007457,"22,433.23",FAILED,CARD,VND0000001
1,PAYMENT0000002,ACC0004445,BRW0011306,2026-07-23 20:25:20,TXN0000030016,"23,295.11",FAILED,UPI,VND0000004
2,PAYMENT0000003,ACC0003853,BRW0002549,2026-01-11 21:27:46,TXN0000010466,"118,814.97",FAILED,NACH,VND0000013


In [ ]:
# Inspect repeated payment IDs

payment_id_conflicts = (
    duplicate_payment_ids
    .groupby("payment_id")
    .agg(
        rows=("payment_id", "size"),
        accounts=("account_id", "nunique"),
        borrowers=("borrower_id", "nunique"),
        references=("payment_reference", "nunique"),
        amounts=("amount", "nunique"),
        statuses=("payment_status", "nunique"),
        timestamps=("event_at", "nunique"),
        methods=("payment_method", "nunique"),
        providers=("provider_id", "nunique")
    )
    .reset_index()
)

display(payment_id_conflicts.head(30))

,payment_id,rows,accounts,borrowers,references,amounts,statuses,timestamps,methods,providers
0,PAYMENT0000056,2,1,1,1,1,1,1,1,1
1,PAYMENT0000076,2,1,1,1,1,1,1,1,1
2,PAYMENT0000108,2,1,1,1,1,1,1,1,1
3,PAYMENT0000149,2,1,1,1,1,1,1,1,1
4,PAYMENT0000199,2,1,1,1,1,1,1,1,1
5,PAYMENT0000255,2,1,1,1,1,1,1,1,1
6,PAYMENT0000263,2,1,1,1,1,1,1,1,1
7,PAYMENT0000311,2,1,1,1,1,1,1,1,1
8,PAYMENT0000552,2,1,1,1,1,1,1,1,1
9,PAYMENT0000553,2,1,1,1,1,1,1,1,1


In [ ]:
# Classify each duplicated payment_id as:
# 1. Exact duplicate
# 2. Conflicting duplicate

payment_id_classification = (
    duplicate_payment_ids
    .groupby("payment_id")
    .agg(
        rows=("payment_id", "size"),
        accounts=("account_id", "nunique"),
        borrowers=("borrower_id", "nunique"),
        references=("payment_reference", "nunique"),
        amounts=("amount", "nunique"),
        statuses=("payment_status", "nunique"),
        timestamps=("event_at", "nunique"),
        methods=("payment_method", "nunique"),
        providers=("provider_id", "nunique")
    )
    .reset_index()
)

# A payment_id is considered an exact duplicate if every
# associated field has only one unique value.
check_columns = [
    "accounts",
    "borrowers",
    "references",
    "amounts",
    "statuses",
    "timestamps",
    "methods",
    "providers"
]

payment_id_classification["duplicate_type"] = np.where(
    payment_id_classification[check_columns].eq(1).all(axis=1),
    "Exact duplicate",
    "Conflicting duplicate"
)

display(
    payment_id_classification["duplicate_type"]
    .value_counts()
    .to_frame("payment_ids")
)

,payment_ids
duplicate_type,
Exact duplicate,500


In [ ]:
# Show the conflicting payment IDs

conflicting_payment_ids = payment_id_classification[
    payment_id_classification["duplicate_type"] == "Conflicting duplicate"
]

display(conflicting_payment_ids)

,payment_id,rows,accounts,borrowers,references,amounts,statuses,timestamps,methods,providers,duplicate_type


In [ ]:
# How many rows are associated with the 500 duplicated payment IDs?

duplicate_payment_id_summary = (
    duplicate_payment_ids
    .groupby("payment_id")
    .size()
    .value_counts()
    .sort_index()
)

display(duplicate_payment_id_summary)

,count
2,500


In [ ]:
# Verify whether every duplicated payment_id has completely
# identical records

duplicate_payment_ids_sorted = duplicate_payment_ids.sort_values(
    ["payment_id", "event_at"]
)

# Number of unique complete records within each payment_id
duplicate_record_versions = (
    duplicate_payment_ids_sorted
    .groupby("payment_id")
    .apply(lambda x: len(x.drop_duplicates()))
    .value_counts()
    .sort_index()
)

display(duplicate_record_versions)

/tmp/ipykernel_1352/3038608033.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: len(x.drop_duplicates()))


,count
1,486
2,14


In [ ]:
# Get the 14 payment IDs whose two records are not identical

conflicting_payment_ids = (
    payment_id_classification[
        payment_id_classification["duplicate_type"] == "Conflicting duplicate"
    ]["payment_id"]
    .tolist()
)

print("Conflicting payment IDs:", len(conflicting_payment_ids))
print(conflicting_payment_ids)

Conflicting payment IDs: 0
[]


In [ ]:
# Show the actual records for those 14 payment IDs

conflicting_records = (
    payments[
        payments["payment_id"].isin(conflicting_payment_ids)
    ]
    .sort_values(["payment_id", "event_at"])
)

display(conflicting_records)

,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id


In [ ]:
# NULL-safe comparison of duplicated payment records

payment_columns = [
    "account_id",
    "borrower_id",
    "event_at",
    "payment_reference",
    "amount",
    "payment_status",
    "payment_method",
    "provider_id"
]

def null_safe_unique_count(series):
    # Convert missing values to a common marker before counting
    return series.astype("object").where(series.notna(), "<NULL>").nunique()

payment_id_check = (
    duplicate_payment_ids
    .groupby("payment_id")
    .agg(
        rows=("payment_id", "size"),
        accounts=("account_id", null_safe_unique_count),
        borrowers=("borrower_id", null_safe_unique_count),
        event_at=("event_at", null_safe_unique_count),
        references=("payment_reference", null_safe_unique_count),
        amounts=("amount", null_safe_unique_count),
        statuses=("payment_status", null_safe_unique_count),
        methods=("payment_method", null_safe_unique_count),
        providers=("provider_id", null_safe_unique_count)
    )
    .reset_index()
)

check_columns = [
    "accounts",
    "borrowers",
    "event_at",
    "references",
    "amounts",
    "statuses",
    "methods",
    "providers"
]

payment_id_check["duplicate_type"] = np.where(
    payment_id_check[check_columns].eq(1).all(axis=1),
    "Exact duplicate",
    "Conflicting duplicate"
)

display(
    payment_id_check["duplicate_type"]
    .value_counts()
    .to_frame("payment_ids")
)

,payment_ids
duplicate_type,
Exact duplicate,486
Conflicting duplicate,14


In [ ]:
conflicting_payment_ids = payment_id_check.loc[
    payment_id_check["duplicate_type"] == "Conflicting duplicate",
    "payment_id"
].tolist()

print("Conflicting payment IDs:", len(conflicting_payment_ids))
print(conflicting_payment_ids)

Conflicting payment IDs: 14
['PAYMENT0001390', 'PAYMENT0002098', 'PAYMENT0002303', 'PAYMENT0005191', 'PAYMENT0007582', 'PAYMENT0007751', 'PAYMENT0008752', 'PAYMENT0010558', 'PAYMENT0016862', 'PAYMENT0018134', 'PAYMENT0020773', 'PAYMENT0021491', 'PAYMENT0022165', 'PAYMENT0023164']


In [ ]:
conflicting_records = (
    payments[
        payments["payment_id"].isin(conflicting_payment_ids)
    ]
    .sort_values(["payment_id", "event_at"])
)

display(conflicting_records)

,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
1389,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,TXN0000046535,"85,205.16",SUCCESS,CASH,VND0000015
25043,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,NaN,"85,205.16",SUCCESS,CASH,VND0000015
2097,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,TXN0000001302,"5,570.92",SUCCESS,NETBANKING,VND0000001
25055,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,NaN,"5,570.92",SUCCESS,NETBANKING,VND0000001
2302,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,TXN0000067311,"59,429.86",SUCCESS,CARD,VND0000011
25057,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,NaN,"59,429.86",SUCCESS,CARD,VND0000011
5190,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,NaN,"107,235.47",SUCCESS,NACH,VND0000002
25437,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,TXN0000041890,"107,235.47",SUCCESS,NACH,VND0000002
7581,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,NaN,"119,714.20",SUCCESS,NETBANKING,VND0000014
25385,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,TXN0000013106,"119,714.20",SUCCESS,NETBANKING,VND0000014


In [ ]:
conflicting_payment_ids = payment_id_check.loc[
    payment_id_check["duplicate_type"] == "Conflicting duplicate",
    "payment_id"
].tolist()

print("Conflicting payment IDs:", len(conflicting_payment_ids))
print(conflicting_payment_ids)

Conflicting payment IDs: 14
['PAYMENT0001390', 'PAYMENT0002098', 'PAYMENT0002303', 'PAYMENT0005191', 'PAYMENT0007582', 'PAYMENT0007751', 'PAYMENT0008752', 'PAYMENT0010558', 'PAYMENT0016862', 'PAYMENT0018134', 'PAYMENT0020773', 'PAYMENT0021491', 'PAYMENT0022165', 'PAYMENT0023164']


In [ ]:
conflicting_records = (
    payments[
        payments["payment_id"].isin(conflicting_payment_ids)
    ]
    .sort_values(["payment_id", "event_at"])
)

display(conflicting_records)

,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
1389,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,TXN0000046535,"85,205.16",SUCCESS,CASH,VND0000015
25043,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,NaN,"85,205.16",SUCCESS,CASH,VND0000015
2097,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,TXN0000001302,"5,570.92",SUCCESS,NETBANKING,VND0000001
25055,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,NaN,"5,570.92",SUCCESS,NETBANKING,VND0000001
2302,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,TXN0000067311,"59,429.86",SUCCESS,CARD,VND0000011
25057,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,NaN,"59,429.86",SUCCESS,CARD,VND0000011
5190,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,NaN,"107,235.47",SUCCESS,NACH,VND0000002
25437,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,TXN0000041890,"107,235.47",SUCCESS,NACH,VND0000002
7581,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,NaN,"119,714.20",SUCCESS,NETBANKING,VND0000014
25385,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,TXN0000013106,"119,714.20",SUCCESS,NETBANKING,VND0000014


In [ ]:
# Quantify the monetary impact of duplicate payment_id records

exact_duplicate_ids = payment_id_check.loc[
    payment_id_check["duplicate_type"] == "Exact duplicate",
    "payment_id"
]

exact_duplicate_records = payments[
    payments["payment_id"].isin(exact_duplicate_ids)
].sort_values(["payment_id", "event_at"])

duplicate_payment_amount = (
    exact_duplicate_records
    .groupby("payment_id")["amount"]
    .first()
    .sum()
)

print(
    f"Amount potentially duplicated by repeated payment IDs: "
    f"₹{duplicate_payment_amount:,.2f}"
)

Amount potentially duplicated by repeated payment IDs: ₹37,299,019.80


In [ ]:
successful_payment_amount = payments.loc[
    payments["payment_status"] == "SUCCESS",
    "amount"
].sum()

print(
    f"Raw successful payment amount: "
    f"₹{successful_payment_amount:,.2f}"
)

print(
    f"Potential duplicate share of successful payment value: "
    f"{duplicate_payment_amount / successful_payment_amount * 100:.2f}%"
)

Raw successful payment amount: ₹1,341,485,926.33
Potential duplicate share of successful payment value: 2.78%


In [ ]:
### 2.2 Payment Reference Forensics

`payment_reference` is another candidate transaction identifier.

The objective is to determine whether repeated payment references represent:

- legitimate reuse,
- multiple payment events,
- retries,
- or duplicate/ambiguous records.

Because recovery is ultimately measured from payments, the reliability of this identifier directly affects the reported recovery amount.

In [ ]:
# Investigate repeated payment references

reference_counts = (
    payments
    .groupby("payment_reference", dropna=False)
    .size()
    .reset_index(name="rows")
)

duplicate_references = reference_counts[
    (reference_counts["rows"] > 1) &
    (reference_counts["payment_reference"].notna())
]

print("Unique payment references:", payments["payment_reference"].nunique())
print("Repeated payment references:", len(duplicate_references))
print(
    "Rows associated with repeated references:",
    duplicate_references["rows"].sum()
)

display(
    duplicate_references
    .sort_values("rows", ascending=False)
    .head(20)
)

Unique payment references: 20821
Repeated payment references: 3745
Rows associated with repeated references: 8042


,payment_reference,rows
13081,TXN0000044312,5
1993,TXN0000006936,5
14862,TXN0000050468,5
15513,TXN0000052559,5
585,TXN0000002005,5
6280,TXN0000021482,5
19524,TXN0000065723,5
16926,TXN0000057390,4
3869,TXN0000013291,4
8063,TXN0000027440,4


In [ ]:
# Examine whether repeated references map to multiple accounts,
# amounts, statuses, or payment IDs

repeated_reference_values = duplicate_references["payment_reference"].tolist()

reference_forensics = (
    payments[
        payments["payment_reference"].isin(repeated_reference_values)
    ]
    .groupby("payment_reference")
    .agg(
        rows=("payment_reference", "size"),
        payment_ids=("payment_id", "nunique"),
        accounts=("account_id", "nunique"),
        borrowers=("borrower_id", "nunique"),
        amounts=("amount", "nunique"),
        statuses=("payment_status", "nunique"),
        timestamps=("event_at", "nunique"),
        methods=("payment_method", "nunique"),
        providers=("provider_id", "nunique")
    )
    .reset_index()
)

display(
    reference_forensics
    .sort_values(
        ["accounts", "amounts", "payment_ids"],
        ascending=False
    )
    .head(30)
)

,payment_reference,rows,payment_ids,accounts,borrowers,amounts,statuses,timestamps,methods,providers
2380,TXN0000044312,5,5,5,5,5,2,5,4,3
88,TXN0000001778,4,4,4,4,4,2,4,3,3
108,TXN0000002005,5,4,4,4,4,4,4,3,4
375,TXN0000006936,5,4,4,4,4,3,4,3,3
511,TXN0000009490,4,4,4,4,4,1,4,3,4
644,TXN0000011850,4,4,4,4,4,2,4,3,3
708,TXN0000013291,4,4,4,4,4,2,4,4,4
1080,TXN0000020022,4,4,4,4,4,3,4,2,4
1090,TXN0000020227,4,4,4,4,4,1,4,3,3
1170,TXN0000021482,5,4,4,4,4,2,4,4,4


In [ ]:
display(
    reference_forensics[
        [
            "payment_reference",
            "rows",
            "payment_ids",
            "accounts",
            "borrowers",
            "amounts",
            "statuses",
            "timestamps",
            "methods",
            "providers"
        ]
    ]
    .sort_values(
        ["accounts", "amounts", "payment_ids"],
        ascending=False
    )
    .head(20)
)

,payment_reference,rows,payment_ids,accounts,borrowers,amounts,statuses,timestamps,methods,providers
2380,TXN0000044312,5,5,5,5,5,2,5,4,3
88,TXN0000001778,4,4,4,4,4,2,4,3,3
108,TXN0000002005,5,4,4,4,4,4,4,3,4
375,TXN0000006936,5,4,4,4,4,3,4,3,3
511,TXN0000009490,4,4,4,4,4,1,4,3,4
644,TXN0000011850,4,4,4,4,4,2,4,3,3
708,TXN0000013291,4,4,4,4,4,2,4,4,4
1080,TXN0000020022,4,4,4,4,4,3,4,2,4
1090,TXN0000020227,4,4,4,4,4,1,4,3,3
1170,TXN0000021482,5,4,4,4,4,2,4,4,4


In [ ]:
### 2.3 Timestamp and Timezone Forensics

Collection events originate from systems using different timezone representations.

The objective is to determine whether timestamps are consistently represented and whether timezone differences could change the business date or hour assigned to an event.

This is particularly important for metrics such as calling time, daily targeting, hourly contact rates, and campaign attribution.

In [ ]:
# Inspect timestamp columns across the major event tables

timestamp_columns = {
    "accounts": ["created_at", "updated_at"],
    "calls": ["event_at"],
    "call_attempts": ["event_at"],
    "whatsapp_events": ["event_at"],
    "sms_events": ["event_at"],
    "field_visits": ["event_at"],
    "promises_to_pay": ["event_at"],
    "payments": ["event_at"],
    "daily_targeting": ["target_date"]
}

for table_name, cols in timestamp_columns.items():
    df = globals()[table_name]

    print(f"\n--- {table_name} ---")

    for col in cols:
        if col in df.columns:
            print(
                f"{col}: dtype={df[col].dtype}, "
                f"min={df[col].min()}, max={df[col].max()}"
            )


--- accounts ---

--- calls ---
event_at: dtype=object, min=2025-12-29 06:52:37, max=2026-08-12 15:43:05

--- call_attempts ---
event_at: dtype=object, min=2026-01-01 00:01:18, max=2026-08-08 23:58:40

--- whatsapp_events ---
event_at: dtype=object, min=2026-01-01 00:01:49, max=2026-08-08 23:56:16

--- sms_events ---
event_at: dtype=object, min=2026-01-01 00:04:16, max=2026-08-08 23:46:06

--- field_visits ---
event_at: dtype=object, min=2026-01-01 00:07:53, max=2026-08-08 23:50:03

--- promises_to_pay ---
event_at: dtype=object, min=2026-01-01 00:24:14, max=2026-08-08 23:59:33

--- payments ---
event_at: dtype=object, min=2026-01-01 00:14:40, max=2026-08-08 23:50:23

--- daily_targeting ---
target_date: dtype=object, min=2026-01-01, max=2026-08-08


In [ ]:
# Inspect all columns for calls, accounts and payments

print("ACCOUNTS COLUMNS")
print(accounts.columns.tolist())

print("\nCALLS COLUMNS")
print(calls.columns.tolist())

print("\nPAYMENTS COLUMNS")
print(payments.columns.tolist())

ACCOUNTS COLUMNS
['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version']

CALLS COLUMNS
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone']

PAYMENTS COLUMNS
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']


In [ ]:
# Timezone distribution for accounts and calls

print("ACCOUNT TIMEZONES")
print(accounts["timezone"].value_counts(dropna=False))

print("\nCALL TIMEZONES")
print(calls["timezone"].value_counts(dropna=False))

ACCOUNT TIMEZONES
timezone
UTC             10096
Asia/Kolkata     9981
Asia/Dubai       9923
Name: count, dtype: int64

CALL TIMEZONES
timezone
Asia/Kolkata    30485
Asia/Dubai      30464
UTC             30401
Name: count, dtype: int64


In [ ]:
# Check whether call timestamps are close to midnight.
# These are the events most likely to cross a calendar day
# if interpreted under a different timezone.

calls_time = pd.to_datetime(calls["event_at"], errors="coerce")

hour_counts = calls_time.dt.hour.value_counts().sort_index()

print("Call volume by hour:")
display(hour_counts.to_frame("calls"))

print(
    "\nCalls occurring between 22:00 and 23:59:",
    ((calls_time.dt.hour >= 22)).sum()
)

print(
    "Calls occurring between 00:00 and 01:59:",
    ((calls_time.dt.hour <= 1)).sum()
)

Call volume by hour:


,calls
event_at,
0,3688
1,3799
2,3942
3,3831
4,3751
5,3864
6,3871
7,3850
8,3765



Calls occurring between 22:00 and 23:59: 7676
Calls occurring between 00:00 and 01:59: 7487


In [ ]:
# Test the impact of timezone normalization on call date/hour

calls_tz = calls[["call_id", "event_at", "timezone"]].copy()

calls_tz["event_local"] = pd.to_datetime(
    calls_tz["event_at"],
    errors="coerce"
)

# Treat event_at as local time in the timezone supplied by the row,
# then convert to UTC.
def to_utc(row):
    try:
        return (
            row["event_local"]
            .tz_localize(row["timezone"])
            .tz_convert("UTC")
        )
    except Exception:
        return pd.NaT

calls_tz["event_utc"] = calls_tz.apply(to_utc, axis=1)

calls_tz["local_date"] = calls_tz["event_local"].dt.date
calls_tz["utc_date"] = calls_tz["event_utc"].dt.date

calls_tz["local_hour"] = calls_tz["event_local"].dt.hour
calls_tz["utc_hour"] = calls_tz["event_utc"].dt.hour

date_changed = calls_tz["local_date"] != calls_tz["utc_date"]
hour_changed = calls_tz["local_hour"] != calls_tz["utc_hour"]

print("Total calls:", len(calls_tz))
print("Calls with date changed after UTC conversion:", date_changed.sum())
print("Calls with hour changed after UTC conversion:", hour_changed.sum())

print(
    "Date-change percentage:",
    f"{date_changed.mean() * 100:.2f}%"
)

print(
    "Hour-change percentage:",
    f"{hour_changed.mean() * 100:.2f}%"
)

Total calls: 91350
Calls with date changed after UTC conversion: 12097
Calls with hour changed after UTC conversion: 60949
Date-change percentage: 13.24%
Hour-change percentage: 66.72%


In [ ]:
display(
    calls_tz.loc[
        date_changed,
        [
            "call_id",
            "event_local",
            "timezone",
            "event_utc",
            "local_date",
            "utc_date",
            "local_hour",
            "utc_hour"
        ]
    ].head(20)
)

,call_id,event_local,timezone,event_utc,local_date,utc_date,local_hour,utc_hour
2,CALL0000003,2026-04-07 00:35:35,Asia/Dubai,2026-04-06 20:35:35+00:00,2026-04-07,2026-04-06,0,20
20,CALL0000021,2026-02-14 04:38:51,Asia/Kolkata,2026-02-13 23:08:51+00:00,2026-02-14,2026-02-13,4,23
28,CALL0000029,2026-03-24 00:14:47,Asia/Dubai,2026-03-23 20:14:47+00:00,2026-03-24,2026-03-23,0,20
36,CALL0000037,2026-04-20 01:18:52,Asia/Kolkata,2026-04-19 19:48:52+00:00,2026-04-20,2026-04-19,1,19
38,CALL0000039,2026-04-28 03:56:21,Asia/Kolkata,2026-04-27 22:26:21+00:00,2026-04-28,2026-04-27,3,22
42,CALL0000043,2026-03-05 03:21:54,Asia/Dubai,2026-03-04 23:21:54+00:00,2026-03-05,2026-03-04,3,23
48,CALL0000049,2026-05-05 04:32:55,Asia/Kolkata,2026-05-04 23:02:55+00:00,2026-05-05,2026-05-04,4,23
50,CALL0000051,2026-05-21 03:10:57,Asia/Dubai,2026-05-20 23:10:57+00:00,2026-05-21,2026-05-20,3,23
72,CALL0000073,2026-04-08 05:17:48,Asia/Kolkata,2026-04-07 23:47:48+00:00,2026-04-08,2026-04-07,5,23
87,CALL0000088,2026-06-16 03:06:13,Asia/Kolkata,2026-06-15 21:36:13+00:00,2026-06-16,2026-06-15,3,21


In [ ]:
# Date changes by source timezone

timezone_date_impact = (
    calls_tz
    .assign(date_changed=date_changed)
    .groupby("timezone")
    .agg(
        calls=("call_id", "count"),
        date_changed=("date_changed", "sum")
    )
    .reset_index()
)

timezone_date_impact["date_change_pct"] = (
    timezone_date_impact["date_changed"]
    / timezone_date_impact["calls"]
    * 100
)

display(timezone_date_impact)

,timezone,calls,date_changed,date_change_pct
0,Asia/Dubai,30464,5039,16.54
1,Asia/Kolkata,30485,7058,23.15
2,UTC,30401,0,0.00


In [ ]:
### 2.4 Agent Identity Forensics

Agent-level productivity metrics depend on correctly identifying the same agent across operational systems.

The objective is to determine whether agent identifiers are consistent across the agent master, calls, call attempts, sessions, and other collection activity.

We will identify identifier collisions and determine the safest canonical agent identifier for the Golden Dataset.

In [ ]:
# Inspect agent identifiers across the relevant datasets

agent_tables = {
    "agents": agents,
    "agent_sessions": agent_sessions,
    "calls": calls,
    "call_attempts": call_attempts
}

for name, df in agent_tables.items():
    agent_cols = [
        c for c in df.columns
        if "agent" in c.lower() or "employee" in c.lower()
    ]

    print(f"\n--- {name} ---")
    print(agent_cols)

    for col in agent_cols:
        print(f"{col}: {df[col].nunique(dropna=True)} unique")


--- agents ---
['agent_id', 'employee_code', 'agent_name']
agent_id: 1000 unique
employee_code: 1099 unique
agent_name: 10 unique

--- agent_sessions ---
['agent_id']
agent_id: 1000 unique

--- calls ---
['agent_id']
agent_id: 1000 unique

--- call_attempts ---
['agent_id']
agent_id: 1000 unique


In [ ]:
# Check whether one agent_id maps to multiple employee_codes

agent_identity = (
    agents
    .groupby("agent_id")
    .agg(
        employee_codes=("employee_code", "nunique"),
        names=("agent_name", "nunique")
    )
    .reset_index()
)

print(
    "Agent IDs mapped to multiple employee codes:",
    (agent_identity["employee_codes"] > 1).sum()
)

print(
    "Agent IDs mapped to multiple names:",
    (agent_identity["names"] > 1).sum()
)

display(
    agent_identity[
        (agent_identity["employee_codes"] > 1) |
        (agent_identity["names"] > 1)
    ].head(20)
)

Agent IDs mapped to multiple employee codes: 1000
Agent IDs mapped to multiple names: 1000


,agent_id,employee_codes,names
0,AGT0000001,23,10
1,AGT0000002,23,10
2,AGT0000003,28,10
3,AGT0000004,28,10
4,AGT0000005,29,10
5,AGT0000006,37,10
6,AGT0000007,29,10
7,AGT0000008,29,10
8,AGT0000009,24,10
9,AGT0000010,26,10


In [ ]:
# Inspect how many records exist per agent_id

agent_record_counts = (
    agents
    .groupby("agent_id")
    .size()
    .value_counts()
    .sort_index()
)

display(
    agent_record_counts.to_frame("agent_ids")
)

,agent_ids
14,4
15,1
17,2
18,9
19,7
20,14
21,19
22,29
23,25
24,50


In [ ]:
# Inspect one agent with multiple identity records

example_agent = (
    agents["agent_id"]
    .value_counts()
    .idxmax()
)

print("Example agent:", example_agent)

display(
    agents[
        agents["agent_id"] == example_agent
    ].sort_values(
        [c for c in ["created_at", "updated_at"] if c in agents.columns]
    )
)

Example agent: AGT0000875


,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
10042,AGT0000875,EMP00565,Priya Mehta,VND0000001,T3,INACTIVE,2024-07-28 03:31:48,2025-01-10 12:24:35
13864,AGT0000875,EMP00090,Aarav Sharma,VND0000015,FIELD,INACTIVE,2024-02-24 04:15:28,2025-01-11 03:47:15
14438,AGT0000875,EMP00771,Pooja Nair,VND0000006,FIELD,SUSPENDED,2024-07-15 23:24:17,2025-01-26 00:39:22
11191,AGT0000875,EMP01030,Neha Singh,VND0000015,T1,SUSPENDED,2025-08-10 11:56:24,2025-01-28 17:03:12
8105,AGT0000875,EMP00570,Ananya Rao,VND0000007,T2,INACTIVE,2025-07-30 04:27:28,2025-03-20 08:49:00
13354,AGT0000875,EMP00602,Amit Kumar,VND0000012,T1,INACTIVE,2024-06-09 08:08:14,2025-04-05 22:53:15
24583,AGT0000875,EMP00109,Neha Singh,VND0000014,T3,INACTIVE,2024-09-03 08:39:54,2025-04-07 04:08:46
13336,AGT0000875,EMP00209,Sneha Das,VND0000013,DIGITAL,ACTIVE,2025-07-02 17:40:18,2025-04-23 07:57:22
11016,AGT0000875,EMP00088,Ananya Rao,VND0000008,T3,SUSPENDED,2024-06-16 23:13:10,2025-04-27 15:15:49
14526,AGT0000875,EMP00588,Neha Singh,VND0000011,T3,ACTIVE,2024-06-06 17:52:42,2025-05-02 00:45:14


In [48]:
# Quantify identity/history records per agent

agent_history_summary = (
    agents
    .groupby("agent_id")
    .agg(
        records=("agent_id", "size"),
        employee_codes=("employee_code", "nunique"),
        names=("agent_name", "nunique"),
        vendors=("vendor_id", "nunique"),
        teams=("team", "nunique"),
        statuses=("status", "nunique")
    )
    .reset_index()
)

display(agent_history_summary.describe())

,records,employee_codes,names,vendors,teams,statuses
count,"1,000.00","1,000.00","1,000.00","1,000.00","1,000.00","1,000.00"
mean,30.00,29.60,9.49,13.01,4.99,3.00
std,5.57,5.46,0.72,1.32,0.10,0.00
min,14.00,14.00,6.00,7.00,4.00,3.00
25%,26.00,26.00,9.00,12.00,5.00,3.00
50%,30.00,29.00,10.00,13.00,5.00,3.00
75%,34.00,33.00,10.00,14.00,5.00,3.00
max,48.00,48.00,10.00,15.00,5.00,3.00


In [49]:
# How many records are there in total and what is the maximum per agent?

print("Total agent records:", len(agents))
print("Unique agents:", agents["agent_id"].nunique())
print("Maximum records for one agent:", agent_history_summary["records"].max())

display(
    agent_history_summary
    .sort_values("records", ascending=False)
    .head(10)
)

Total agent records: 30000
Unique agents: 1000
Maximum records for one agent: 48


,agent_id,records,employee_codes,names,vendors,teams,statuses
532,AGT0000533,48,47,10,15,5,3
874,AGT0000875,48,47,10,13,5,3
366,AGT0000367,48,48,10,15,5,3
842,AGT0000843,47,46,10,14,5,3
539,AGT0000540,46,45,10,15,5,3
875,AGT0000876,46,45,10,14,4,3
542,AGT0000543,45,45,10,15,5,3
562,AGT0000563,45,45,10,15,5,3
439,AGT0000440,45,44,10,13,5,3
882,AGT0000883,44,43,10,13,5,3


In [49]:
### 2.5 Denominator Manipulation

The objective is to determine whether unsuccessful accounts remain in the analytical population throughout the collection funnel.

A reported conversion improvement can be artificially created if accounts with unsuccessful outcomes are excluded from the denominator.

We therefore compare the account population across targeting, collection activity, contact, PTP, and payment stages.

In [50]:
# Basic account population at each collection stage

eligible_accounts = accounts["account_id"].nunique()

targeted_accounts = daily_targeting["account_id"].nunique()
attempted_accounts = call_attempts["account_id"].nunique()
called_accounts = calls["account_id"].nunique()
ptp_accounts = promises_to_pay["account_id"].nunique()
payment_accounts = payments[
    payments["payment_status"] == "SUCCESS"
]["account_id"].nunique()

funnel = pd.DataFrame({
    "Stage": [
        "All accounts",
        "Targeted",
        "Call attempted",
        "Called",
        "PTP created",
        "Successful payment"
    ],
    "Unique accounts": [
        eligible_accounts,
        targeted_accounts,
        attempted_accounts,
        called_accounts,
        ptp_accounts,
        payment_accounts
    ]
})

display(funnel)

,Stage,Unique accounts
0,All accounts,30000
1,Targeted,23344
2,Call attempted,29451
3,Called,28408
4,PTP created,13532
5,Successful payment,13284


In [51]:
# Coverage relative to the total account population

funnel["Coverage %"] = (
    funnel["Unique accounts"]
    / eligible_accounts
    * 100
)

display(funnel)

,Stage,Unique accounts,Coverage %
0,All accounts,30000,100.00
1,Targeted,23344,77.81
2,Call attempted,29451,98.17
3,Called,28408,94.69
4,PTP created,13532,45.11
5,Successful payment,13284,44.28


In [52]:
targeted_ids = set(daily_targeting["account_id"].dropna().unique())
attempted_ids = set(call_attempts["account_id"].dropna().unique())

attempted_not_targeted = attempted_ids - targeted_ids

print("Attempted accounts:", len(attempted_ids))
print("Targeted accounts:", len(targeted_ids))
print("Attempted but not targeted:", len(attempted_not_targeted))

Attempted accounts: 29451
Targeted accounts: 23344
Attempted but not targeted: 6526


In [53]:
# What happened to the attempted-but-not-targeted accounts?

attempted_not_targeted_df = call_attempts[
    call_attempts["account_id"].isin(attempted_not_targeted)
]

display(
    attempted_not_targeted_df[
        ["account_id", "event_at", "agent_id"]
    ].head(20)
)

,account_id,event_at,agent_id
0,ACC0009527,2026-03-10 21:31:13,AGT0000685
7,ACC0016760,2026-04-10 20:44:44,AGT0000638
11,ACC0014145,2026-05-09 00:05:52,AGT0000188
20,ACC0003922,2026-01-18 20:30:55,AGT0000320
21,ACC0019806,2026-05-03 02:36:25,AGT0000910
25,ACC0022758,2026-04-30 12:19:20,AGT0000949
26,ACC0000123,2026-01-01 19:48:57,AGT0000680
27,ACC0020094,2026-06-03 22:24:50,AGT0000213
34,ACC0007422,2026-08-05 08:12:40,AGT0000965
41,ACC0010726,2026-05-20 17:45:08,AGT0000037


In [54]:
# Month-by-month account population

targeting_monthly = daily_targeting.copy()
targeting_monthly["target_date"] = pd.to_datetime(
    targeting_monthly["target_date"],
    errors="coerce"
)

targeting_monthly["month"] = (
    targeting_monthly["target_date"]
    .dt.to_period("M")
)

monthly_denominator = (
    targeting_monthly
    .groupby("month")
    .agg(
        targeted_accounts=("account_id", "nunique")
    )
    .reset_index()
)

payments_monthly = payments.copy()
payments_monthly["event_at"] = pd.to_datetime(
    payments_monthly["event_at"],
    errors="coerce"
)

payments_monthly["month"] = (
    payments_monthly["event_at"]
    .dt.to_period("M")
)

successful_payments_monthly = (
    payments_monthly[
        payments_monthly["payment_status"] == "SUCCESS"
    ]
    .groupby("month")
    .agg(
        paying_accounts=("account_id", "nunique"),
        recovery=("amount", "sum")
    )
    .reset_index()
)

denominator_analysis = monthly_denominator.merge(
    successful_payments_monthly,
    on="month",
    how="outer"
).sort_values("month")

denominator_analysis["recovery_per_targeted_account"] = (
    denominator_analysis["recovery"]
    / denominator_analysis["targeted_accounts"]
)

display(denominator_analysis)

,month,targeted_accounts,paying_accounts,recovery,recovery_per_targeted_account
0,2026-01,5732,2374,"191,133,284.42","33,344.96"
1,2026-02,5160,2173,"174,097,287.96","33,739.78"
2,2026-03,5666,2419,"193,233,384.29","34,104.02"
3,2026-04,5585,2304,"178,427,017.05","31,947.54"
4,2026-05,5800,2344,"187,048,144.34","32,249.68"
5,2026-06,5535,2286,"178,724,493.46","32,289.88"
6,2026-07,5666,2335,"190,278,846.88","33,582.57"
7,2026-08,1566,608,"48,543,467.93","30,998.38"


In [ ]:
## 3. Is the Reported 11% Improvement Real?

We compare the reported month-on-month improvement against independently reconstructed recovery measures.

Recovery is evaluated under multiple defensible definitions:

1. Raw successful payment amount
2. Payment-ID deduplicated successful payment amount
3. Recovery per targeted account

The objective is to determine whether the reported 11% improvement is robust to reasonable data-cleaning and denominator choices.

In [55]:
# Prepare payment data for independent recovery calculations

payments_analysis = payments.copy()

payments_analysis["event_at"] = pd.to_datetime(
    payments_analysis["event_at"],
    errors="coerce"
)

payments_analysis["month"] = (
    payments_analysis["event_at"]
    .dt.to_period("M")
)

successful_payments = payments_analysis[
    payments_analysis["payment_status"] == "SUCCESS"
].copy()

# Raw recovery
raw_monthly = (
    successful_payments
    .groupby("month")["amount"]
    .sum()
    .rename("raw_recovery")
)

# Payment-ID deduplicated recovery
dedup_successful_payments = (
    successful_payments
    .drop_duplicates("payment_id")
)

dedup_monthly = (
    dedup_successful_payments
    .groupby("month")["amount"]
    .sum()
    .rename("dedup_recovery")
)

recovery_comparison = pd.concat(
    [raw_monthly, dedup_monthly],
    axis=1
).reset_index()

# Add targeted-account denominator
recovery_comparison = recovery_comparison.merge(
    monthly_denominator[
        ["month", "targeted_accounts"]
    ],
    on="month",
    how="left"
)

recovery_comparison["raw_recovery_per_targeted"] = (
    recovery_comparison["raw_recovery"]
    / recovery_comparison["targeted_accounts"]
)

recovery_comparison["dedup_recovery_per_targeted"] = (
    recovery_comparison["dedup_recovery"]
    / recovery_comparison["targeted_accounts"]
)

display(recovery_comparison)

,month,raw_recovery,dedup_recovery,targeted_accounts,raw_recovery_per_targeted,dedup_recovery_per_targeted
0,2026-01,"191,133,284.42","187,229,127.59",5732,"33,344.96","32,663.84"
1,2026-02,"174,097,287.96","170,142,453.76",5160,"33,739.78","32,973.34"
2,2026-03,"193,233,384.29","188,912,374.02",5666,"34,104.02","33,341.40"
3,2026-04,"178,427,017.05","175,138,043.41",5585,"31,947.54","31,358.65"
4,2026-05,"187,048,144.34","184,250,278.50",5800,"32,249.68","31,767.29"
5,2026-06,"178,724,493.46","175,559,726.97",5535,"32,289.88","31,718.11"
6,2026-07,"190,278,846.88","187,242,265.08",5666,"33,582.57","33,046.64"
7,2026-08,"48,543,467.93","47,109,695.31",1566,"30,998.38","30,082.82"


In [56]:
# Prepare payment data for independent recovery calculations

payments_analysis = payments.copy()

payments_analysis["event_at"] = pd.to_datetime(
    payments_analysis["event_at"],
    errors="coerce"
)

payments_analysis["month"] = (
    payments_analysis["event_at"]
    .dt.to_period("M")
)

successful_payments = payments_analysis[
    payments_analysis["payment_status"] == "SUCCESS"
].copy()

# Raw recovery
raw_monthly = (
    successful_payments
    .groupby("month")["amount"]
    .sum()
    .rename("raw_recovery")
)

# Payment-ID deduplicated recovery
dedup_successful_payments = (
    successful_payments
    .drop_duplicates("payment_id")
)

dedup_monthly = (
    dedup_successful_payments
    .groupby("month")["amount"]
    .sum()
    .rename("dedup_recovery")
)

recovery_comparison = pd.concat(
    [raw_monthly, dedup_monthly],
    axis=1
).reset_index()

# Add targeted-account denominator
recovery_comparison = recovery_comparison.merge(
    monthly_denominator[
        ["month", "targeted_accounts"]
    ],
    on="month",
    how="left"
)

recovery_comparison["raw_recovery_per_targeted"] = (
    recovery_comparison["raw_recovery"]
    / recovery_comparison["targeted_accounts"]
)

recovery_comparison["dedup_recovery_per_targeted"] = (
    recovery_comparison["dedup_recovery"]
    / recovery_comparison["targeted_accounts"]
)

display(recovery_comparison)

,month,raw_recovery,dedup_recovery,targeted_accounts,raw_recovery_per_targeted,dedup_recovery_per_targeted
0,2026-01,"191,133,284.42","187,229,127.59",5732,"33,344.96","32,663.84"
1,2026-02,"174,097,287.96","170,142,453.76",5160,"33,739.78","32,973.34"
2,2026-03,"193,233,384.29","188,912,374.02",5666,"34,104.02","33,341.40"
3,2026-04,"178,427,017.05","175,138,043.41",5585,"31,947.54","31,358.65"
4,2026-05,"187,048,144.34","184,250,278.50",5800,"32,249.68","31,767.29"
5,2026-06,"178,724,493.46","175,559,726.97",5535,"32,289.88","31,718.11"
6,2026-07,"190,278,846.88","187,242,265.08",5666,"33,582.57","33,046.64"
7,2026-08,"48,543,467.93","47,109,695.31",1566,"30,998.38","30,082.82"


In [ ]:
## 4. Why Did Recovery Change?

The next step is to determine whether the July improvement reflects:

1. Genuine operational improvement, or
2. A change in the underlying borrower/account population.

We first test portfolio mix using DPD and risk segment before investigating operational channels and campaigns.

In [57]:
# Compare portfolio mix between June and July

accounts_mix = accounts.copy()

accounts_mix["opened_at"] = pd.to_datetime(
    accounts_mix["opened_at"],
    errors="coerce"
)

# Use daily targeting as the monthly population
targeting_mix = daily_targeting.copy()

targeting_mix["target_date"] = pd.to_datetime(
    targeting_mix["target_date"],
    errors="coerce"
)

targeting_mix["month"] = targeting_mix["target_date"].dt.to_period("M")

targeting_with_accounts = targeting_mix.merge(
    accounts_mix[
        [
            "account_id",
            "dpd",
            "risk_segment",
            "loan_type"
        ]
    ],
    on="account_id",
    how="left"
)

mix_summary = (
    targeting_with_accounts[
        targeting_with_accounts["month"].astype(str).isin(
            ["2026-06", "2026-07"]
        )
    ]
    .groupby(["month", "risk_segment"])
    .agg(
        accounts=("account_id", "nunique")
    )
    .reset_index()
)

mix_summary["share_pct"] = (
    mix_summary["accounts"]
    / mix_summary.groupby("month")["accounts"].transform("sum")
    * 100
)

display(mix_summary)

,month,risk_segment,accounts,share_pct
0,2026-06,HIGH,1374,24.82
1,2026-06,LOW,1405,25.38
2,2026-06,MEDIUM,1422,25.69
3,2026-06,NPA,1334,24.10
4,2026-07,HIGH,1421,25.08
5,2026-07,LOW,1424,25.13
6,2026-07,MEDIUM,1424,25.13
7,2026-07,NPA,1397,24.66


In [58]:
# DPD mix

dpd_mix = (
    targeting_with_accounts[
        targeting_with_accounts["month"].astype(str).isin(
            ["2026-06", "2026-07"]
        )
    ]
    .assign(
        dpd_bucket=lambda x: pd.cut(
            x["dpd"],
            bins=[-1, 30, 60, 90, np.inf],
            labels=["0-30", "31-60", "61-90", "90+"]
        )
    )
    .groupby(["month", "dpd_bucket"])
    .agg(
        accounts=("account_id", "nunique")
    )
    .reset_index()
)

dpd_mix["share_pct"] = (
    dpd_mix["accounts"]
    / dpd_mix.groupby("month")["accounts"].transform("sum")
    * 100
)

display(dpd_mix)

/tmp/ipykernel_1352/1145901455.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["month", "dpd_bucket"])


,month,dpd_bucket,accounts,share_pct
0,2026-06,0-30,2455,44.35
1,2026-06,31-60,1003,18.12
2,2026-06,61-90,1080,19.51
3,2026-06,90+,997,18.01
4,2026-07,0-30,2530,44.65
5,2026-07,31-60,1074,18.96
6,2026-07,61-90,1054,18.60
7,2026-07,90+,1008,17.79


In [ ]:
### 4.1 Within-Segment Recovery Performance

Portfolio composition between June and July is broadly similar.

We now compare recovery performance within the same risk and DPD segments to determine whether the improvement remains after controlling for basic portfolio composition.

In [59]:
# Prepare June/July successful payments

payments_jj = successful_payments[
    successful_payments["month"].astype(str).isin(
        ["2026-06", "2026-07"]
    )
].copy()

# Add account-level risk and DPD
payments_jj = payments_jj.merge(
    accounts[
        [
            "account_id",
            "risk_segment",
            "dpd"
        ]
    ],
    on="account_id",
    how="left"
)

payments_jj["dpd_bucket"] = pd.cut(
    payments_jj["dpd"],
    bins=[-1, 30, 60, 90, np.inf],
    labels=["0-30", "31-60", "61-90", "90+"]
)

within_segment = (
    payments_jj
    .groupby(
        ["month", "risk_segment", "dpd_bucket"],
        observed=True
    )
    .agg(
        recovery=("amount", "sum"),
        paying_accounts=("account_id", "nunique")
    )
    .reset_index()
)

display(within_segment)

,month,risk_segment,dpd_bucket,recovery,paying_accounts
0,2026-06,HIGH,0-30,"23,124,900.60",284
1,2026-06,HIGH,31-60,"8,640,262.48",108
2,2026-06,HIGH,61-90,"6,831,725.88",84
3,2026-06,HIGH,90+,"7,591,485.02",100
4,2026-06,LOW,0-30,"17,874,478.63",248
5,2026-06,LOW,31-60,"8,631,951.43",108
6,2026-06,LOW,61-90,"8,319,946.77",118
7,2026-06,LOW,90+,"9,274,977.27",112
8,2026-06,MEDIUM,0-30,"20,054,411.81",270
9,2026-06,MEDIUM,31-60,"9,874,781.49",124


In [60]:
# Targeted-account denominator by risk × DPD segment

targeted_segment = (
    targeting_with_accounts[
        targeting_with_accounts["month"].astype(str).isin(
            ["2026-06", "2026-07"]
        )
    ]
    .assign(
        dpd_bucket=lambda x: pd.cut(
            x["dpd"],
            bins=[-1, 30, 60, 90, np.inf],
            labels=["0-30", "31-60", "61-90", "90+"]
        )
    )
    .groupby(
        ["month", "risk_segment", "dpd_bucket"],
        observed=True
    )
    .agg(
        targeted_accounts=("account_id", "nunique")
    )
    .reset_index()
)

display(targeted_segment)

,month,risk_segment,dpd_bucket,targeted_accounts
0,2026-06,HIGH,0-30,638
1,2026-06,HIGH,31-60,232
2,2026-06,HIGH,61-90,248
3,2026-06,HIGH,90+,256
4,2026-06,LOW,0-30,589
5,2026-06,LOW,31-60,271
6,2026-06,LOW,61-90,295
7,2026-06,LOW,90+,250
8,2026-06,MEDIUM,0-30,644
9,2026-06,MEDIUM,31-60,255


In [61]:
segment_performance = within_segment.merge(
    targeted_segment,
    on=["month", "risk_segment", "dpd_bucket"],
    how="left"
)

segment_performance["recovery_per_targeted_account"] = (
    segment_performance["recovery"]
    / segment_performance["targeted_accounts"]
)

display(
    segment_performance.sort_values(
        ["risk_segment", "dpd_bucket", "month"]
    )
)

,month,risk_segment,dpd_bucket,recovery,paying_accounts,targeted_accounts,recovery_per_targeted_account
0,2026-06,HIGH,0-30,"23,124,900.60",284,638,"36,245.93"
16,2026-07,HIGH,0-30,"20,100,039.70",266,633,"31,753.62"
1,2026-06,HIGH,31-60,"8,640,262.48",108,232,"37,242.51"
17,2026-07,HIGH,31-60,"7,558,978.45",93,287,"26,337.90"
2,2026-06,HIGH,61-90,"6,831,725.88",84,248,"27,547.28"
18,2026-07,HIGH,61-90,"10,262,837.10",122,255,"40,246.42"
3,2026-06,HIGH,90+,"7,591,485.02",100,256,"29,654.24"
19,2026-07,HIGH,90+,"7,311,122.26",99,246,"29,720.01"
4,2026-06,LOW,0-30,"17,874,478.63",248,589,"30,347.16"
20,2026-07,LOW,0-30,"21,776,219.05",261,599,"36,354.29"


In [62]:
# June → July change within each risk × DPD segment

segment_pivot = (
    segment_performance[
        segment_performance["month"].astype(str).isin(
            ["2026-06", "2026-07"]
        )
    ]
    .pivot_table(
        index=["risk_segment", "dpd_bucket"],
        columns="month",
        values="recovery_per_targeted_account"
    )
)

segment_pivot["MoM_change_pct"] = (
    (segment_pivot["2026-07"] / segment_pivot["2026-06"]) - 1
) * 100

display(segment_pivot.sort_values("MoM_change_pct", ascending=False))

/tmp/ipykernel_1352/2638481027.py:9: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


month                     2026-06   2026-07  MoM_change_pct
risk_segment dpd_bucket                                    
NPA          61-90      24,064.78 35,808.36           48.80
HIGH         61-90      27,547.28 40,246.42           46.10
MEDIUM       90+        26,847.90 35,635.96           32.73
LOW          0-30       30,347.16 36,354.29           19.79
             31-60      31,852.22 35,584.05           11.72
             61-90      28,203.21 31,027.64           10.01
             90+        37,099.91 40,547.56            9.29
MEDIUM       0-30       31,140.39 33,561.60            7.78
HIGH         90+        29,654.24 29,720.01            0.22
NPA          31-60      35,990.47 35,780.62           -0.58
             0-30       31,481.57 30,130.51           -4.29
MEDIUM       31-60      38,724.63 35,577.79           -8.13
NPA          90+        34,357.51 31,201.24           -9.19
MEDIUM       61-90      36,186.48 32,699.03           -9.64
HIGH         0-30       36,245.93 31,753.62          -12.39
             31-60      37,242.51 26,337.90          -29.28

In [ ]:
### 4.2 Campaign Performance

The portfolio mix does not explain the full June-to-July movement.

We now compare recovery performance across campaigns to identify whether changes in campaign allocation or campaign effectiveness contributed to the observed improvement.

In [63]:
# Inspect campaign structure

print("Campaigns:")
display(campaigns.head())

print("\nDaily targeting columns:")
print(daily_targeting.columns.tolist())

print("\nNumber of campaigns in targeting:")
print(daily_targeting["campaign_id"].nunique())

Campaigns:


,campaign_id,campaign_name,channel,strategy_version,start_at,target_definition,end_at
0,CMP0000001,DIGITAL_FOLLOWUP,FIELD,legacy,2026-02-17 06:56:01,DPD>=30,2026-04-24 06:56:01
1,CMP0000002,BOUNCE,MIXED,v2,2026-04-30 17:51:31,DPD>=60,2026-05-18 17:51:31
2,CMP0000003,30DPD_W1,MIXED,v1,2026-04-11 16:02:51,DPD>=30,2026-05-18 16:02:51
3,CMP0000004,BOUNCE,VOICE,legacy,2026-04-28 06:15:30,DPD>=60,2026-05-29 06:15:30
4,CMP0000005,60DPD_INTENT,WHATSAPP,legacy,2026-05-04 04:16:21,DPD>=60,2026-07-08 04:16:21



Daily targeting columns:
['target_id', 'account_id', 'campaign_id', 'target_date', 'priority', 'recommended_channel', 'status']

Number of campaigns in targeting:
120


In [64]:
# Build June/July campaign-level targeting population

campaign_targeting = daily_targeting.copy()

campaign_targeting["target_date"] = pd.to_datetime(
    campaign_targeting["target_date"],
    errors="coerce"
)

campaign_targeting["month"] = (
    campaign_targeting["target_date"].dt.to_period("M")
)

campaign_targeting_jj = campaign_targeting[
    campaign_targeting["month"].astype(str).isin(
        ["2026-06", "2026-07"]
    )
]

campaign_population = (
    campaign_targeting_jj
    .groupby(["month", "campaign_id"])
    .agg(
        targeted_accounts=("account_id", "nunique")
    )
    .reset_index()
)

display(campaign_population)

,month,campaign_id,targeted_accounts
0,2026-06,CMP0000001,58
1,2026-06,CMP0000002,61
2,2026-06,CMP0000003,59
3,2026-06,CMP0000004,55
4,2026-06,CMP0000005,58
...,...,...,...
235,2026-07,CMP0000116,51
236,2026-07,CMP0000117,56
237,2026-07,CMP0000118,49
238,2026-07,CMP0000119,50


In [65]:
# Attach successful payment recovery to the campaign that targeted the account

campaign_payment_data = (
    successful_payments[
        successful_payments["month"].astype(str).isin(
            ["2026-06", "2026-07"]
        )
    ][
        ["account_id", "amount", "month"]
    ]
    .merge(
        daily_targeting[
            ["account_id", "campaign_id", "target_date"]
        ],
        on="account_id",
        how="inner"
    )
)

campaign_payment_data["target_date"] = pd.to_datetime(
    campaign_payment_data["target_date"],
    errors="coerce"
)

campaign_payment_data["target_month"] = (
    campaign_payment_data["target_date"].dt.to_period("M")
)

# Keep payments aligned to the campaign's month
campaign_payment_data = campaign_payment_data[
    campaign_payment_data["target_month"] == campaign_payment_data["month"]
]

campaign_recovery = (
    campaign_payment_data
    .groupby(["month", "campaign_id"])
    .agg(
        recovery=("amount", "sum"),
        paying_accounts=("account_id", "nunique")
    )
    .reset_index()
)

campaign_recovery = campaign_recovery.merge(
    campaign_population,
    on=["month", "campaign_id"],
    how="left"
)

campaign_recovery["recovery_per_targeted"] = (
    campaign_recovery["recovery"]
    / campaign_recovery["targeted_accounts"]
)

display(
    campaign_recovery
    .sort_values(
        ["month", "recovery_per_targeted"],
        ascending=[True, False]
    ).head(20)
)

,month,campaign_id,recovery,paying_accounts,targeted_accounts,recovery_per_targeted
77,2026-06,CMP0000083,"895,677.90",8,58,"15,442.72"
13,2026-06,CMP0000015,"647,652.53",7,42,"15,420.30"
106,2026-06,CMP0000112,"648,017.12",8,44,"14,727.66"
107,2026-06,CMP0000113,"643,516.50",5,44,"14,625.38"
101,2026-06,CMP0000107,"576,064.93",7,40,"14,401.62"
98,2026-06,CMP0000104,"791,626.85",9,58,"13,648.74"
12,2026-06,CMP0000014,"602,791.15",7,45,"13,395.36"
73,2026-06,CMP0000079,"598,138.52",9,45,"13,291.97"
68,2026-06,CMP0000074,"664,527.68",6,53,"12,538.26"
38,2026-06,CMP0000041,"618,658.60",6,54,"11,456.64"


In [66]:
campaign_pivot = (
    campaign_recovery
    .pivot_table(
        index="campaign_id",
        columns="month",
        values="recovery_per_targeted"
    )
)

campaign_pivot["MoM_change_pct"] = (
    campaign_pivot["2026-07"]
    / campaign_pivot["2026-06"]
    - 1
) * 100

campaign_pivot = campaign_pivot.sort_values(
    "MoM_change_pct",
    ascending=False
)

display(campaign_pivot.head(10))
display(campaign_pivot.tail(10))

month,2026-06,2026-07,MoM_change_pct
campaign_id,,,
CMP0000055,943.06,"8,919.94",845.85
CMP0000054,"1,061.45","9,910.88",833.72
CMP0000081,"1,795.47","12,694.58",607.03
CMP0000028,"1,472.93","10,283.89",598.19
CMP0000078,884.18,"5,203.46",488.51
CMP0000029,"2,474.69","14,238.87",475.38
CMP0000082,"3,272.34","15,650.42",378.26
CMP0000037,"3,712.30","15,263.95",311.17
CMP0000118,"3,980.36","15,781.85",296.49


month,2026-06,2026-07,MoM_change_pct
campaign_id,,,
CMP0000074,"12,538.26","2,361.50",-81.17
CMP0000111,"8,246.69","1,219.97",-85.21
CMP0000083,"15,442.72","1,072.53",-93.05
CMP0000011,NaN,"4,626.41",NaN
CMP0000016,NaN,"8,084.19",NaN
CMP0000019,"8,155.61",NaN,NaN
CMP0000043,NaN,"6,324.03",NaN
CMP0000053,NaN,"6,181.92",NaN
CMP0000069,NaN,"4,798.54",NaN


In [67]:
# Add campaign attributes to campaign performance

campaign_recovery_enriched = campaign_recovery.merge(
    campaigns[
        [
            "campaign_id",
            "campaign_name",
            "channel",
            "strategy_version",
            "target_definition"
        ]
    ],
    on="campaign_id",
    how="left"
)

display(
    campaign_recovery_enriched[
        [
            "month",
            "campaign_id",
            "campaign_name",
            "channel",
            "strategy_version",
            "target_definition",
            "targeted_accounts",
            "paying_accounts",
            "recovery",
            "recovery_per_targeted"
        ]
    ].head(20)
)

,month,campaign_id,campaign_name,channel,strategy_version,target_definition,targeted_accounts,paying_accounts,recovery,recovery_per_targeted
0,2026-06,CMP0000001,DIGITAL_FOLLOWUP,FIELD,legacy,DPD>=30,58,5,"478,111.51","8,243.30"
1,2026-06,CMP0000002,BOUNCE,MIXED,v2,DPD>=60,61,6,"532,796.19","8,734.36"
2,2026-06,CMP0000003,30DPD_W1,MIXED,v1,DPD>=30,59,6,"347,009.34","5,881.51"
3,2026-06,CMP0000004,BOUNCE,VOICE,legacy,DPD>=60,55,6,"441,256.88","8,022.85"
4,2026-06,CMP0000005,60DPD_INTENT,WHATSAPP,legacy,DPD>=60,58,5,"540,036.79","9,310.98"
5,2026-06,CMP0000006,BOUNCE,MIXED,legacy,DPD>=60,57,6,"453,604.88","7,957.98"
6,2026-06,CMP0000007,DIGITAL_FOLLOWUP,MIXED,v2,PROMISE_BROKEN,52,7,"594,768.50","11,437.86"
7,2026-06,CMP0000008,NPA_RECOVERY,SMS,v2,PROMISE_BROKEN,51,4,"247,490.10","4,852.75"
8,2026-06,CMP0000009,60DPD_INTENT,WHATSAPP,v1,HIGH_RISK,46,2,"132,977.20","2,890.81"
9,2026-06,CMP0000010,30DPD_W1,FIELD,v1,DPD>=30,50,3,"242,644.18","4,852.88"


In [68]:
# Channel-level performance

channel_performance = (
    campaign_recovery_enriched
    .groupby(["month", "channel"])
    .agg(
        targeted_accounts=("targeted_accounts", "sum"),
        recovery=("recovery", "sum"),
        paying_accounts=("paying_accounts", "sum")
    )
    .reset_index()
)

channel_performance["recovery_per_targeted"] = (
    channel_performance["recovery"]
    / channel_performance["targeted_accounts"]
)

display(
    channel_performance.sort_values(
        ["month", "recovery_per_targeted"],
        ascending=[True, False]
    )
)

,month,channel,targeted_accounts,recovery,paying_accounts,recovery_per_targeted
0,2026-06,FIELD,921,"6,916,483.53",88,"7,509.75"
4,2026-06,WHATSAPP,1468,"9,572,758.85",117,"6,520.95"
3,2026-06,VOICE,990,"6,143,423.18",78,"6,205.48"
1,2026-06,MIXED,1153,"6,880,340.84",92,"5,967.34"
2,2026-06,SMS,1389,"7,288,481.06",106,"5,247.29"
5,2026-07,FIELD,939,"6,861,979.17",86,"7,307.75"
6,2026-07,MIXED,1158,"8,192,141.64",104,"7,074.39"
9,2026-07,WHATSAPP,1591,"11,080,849.10",129,"6,964.71"
8,2026-07,VOICE,1078,"7,451,430.03",98,"6,912.27"
7,2026-07,SMS,1369,"9,278,436.27",112,"6,777.53"


In [69]:
# Strategy-version performance

strategy_performance = (
    campaign_recovery_enriched
    .groupby(["month", "strategy_version"])
    .agg(
        targeted_accounts=("targeted_accounts", "sum"),
        recovery=("recovery", "sum"),
        paying_accounts=("paying_accounts", "sum")
    )
    .reset_index()
)

strategy_performance["recovery_per_targeted"] = (
    strategy_performance["recovery"]
    / strategy_performance["targeted_accounts"]
)

display(
    strategy_performance.sort_values(
        ["month", "recovery_per_targeted"],
        ascending=[True, False]
    )
)

,month,strategy_version,targeted_accounts,recovery,paying_accounts,recovery_per_targeted
0,2026-06,legacy,1873,"13,136,302.56",161,"7,013.51"
2,2026-06,v2,1396,"9,030,819.29",118,"6,469.07"
1,2026-06,v1,1149,"6,344,313.40",91,"5,521.60"
3,2026-06,v3,1503,"8,290,052.21",111,"5,515.67"
7,2026-07,v3,1582,"11,839,333.34",143,"7,483.78"
4,2026-07,legacy,1918,"14,038,454.93",172,"7,319.32"
5,2026-07,v1,1220,"8,184,864.11",104,"6,708.91"
6,2026-07,v2,1415,"8,802,183.83",110,"6,220.62"


In [70]:
# Compare strategy allocation between June and July

strategy_mix = (
    strategy_performance.copy()
)

strategy_mix["target_share_pct"] = (
    strategy_mix["targeted_accounts"]
    / strategy_mix.groupby("month")["targeted_accounts"].transform("sum")
    * 100
)

display(
    strategy_mix.sort_values(
        ["month", "target_share_pct"],
        ascending=[True, False]
    )
)

,month,strategy_version,targeted_accounts,recovery,paying_accounts,recovery_per_targeted,target_share_pct
0,2026-06,legacy,1873,"13,136,302.56",161,"7,013.51",31.63
3,2026-06,v3,1503,"8,290,052.21",111,"5,515.67",25.38
2,2026-06,v2,1396,"9,030,819.29",118,"6,469.07",23.58
1,2026-06,v1,1149,"6,344,313.40",91,"5,521.60",19.41
4,2026-07,legacy,1918,"14,038,454.93",172,"7,319.32",31.26
7,2026-07,v3,1582,"11,839,333.34",143,"7,483.78",25.79
6,2026-07,v2,1415,"8,802,183.83",110,"6,220.62",23.06
5,2026-07,v1,1220,"8,184,864.11",104,"6,708.91",19.89


In [71]:
# Compare channel allocation between June and July

channel_mix = (
    channel_performance.copy()
)

channel_mix["target_share_pct"] = (
    channel_mix["targeted_accounts"]
    / channel_mix.groupby("month")["targeted_accounts"].transform("sum")
    * 100
)

display(
    channel_mix.sort_values(
        ["month", "target_share_pct"],
        ascending=[True, False]
    )
)

,month,channel,targeted_accounts,recovery,paying_accounts,recovery_per_targeted,target_share_pct
4,2026-06,WHATSAPP,1468,"9,572,758.85",117,"6,520.95",24.79
2,2026-06,SMS,1389,"7,288,481.06",106,"5,247.29",23.46
1,2026-06,MIXED,1153,"6,880,340.84",92,"5,967.34",19.47
3,2026-06,VOICE,990,"6,143,423.18",78,"6,205.48",16.72
0,2026-06,FIELD,921,"6,916,483.53",88,"7,509.75",15.55
9,2026-07,WHATSAPP,1591,"11,080,849.10",129,"6,964.71",25.93
7,2026-07,SMS,1369,"9,278,436.27",112,"6,777.53",22.31
6,2026-07,MIXED,1158,"8,192,141.64",104,"7,074.39",18.88
8,2026-07,VOICE,1078,"7,451,430.03",98,"6,912.27",17.57
5,2026-07,FIELD,939,"6,861,979.17",86,"7,307.75",15.31


In [72]:
# Estimate the observed v3 improvement and its July recovery impact

v3 = strategy_performance[
    strategy_performance["strategy_version"] == "v3"
].copy()

v3 = v3.set_index("month")

june_v3_rate = v3.loc["2026-06", "recovery_per_targeted"]
july_v3_rate = v3.loc["2026-07", "recovery_per_targeted"]

july_v3_accounts = v3.loc["2026-07", "targeted_accounts"]

v3_observed_incremental_rate = july_v3_rate - june_v3_rate

v3_july_incremental_recovery = (
    v3_observed_incremental_rate * july_v3_accounts
)

print(f"June v3 recovery/targeted account: ₹{june_v3_rate:,.2f}")
print(f"July v3 recovery/targeted account: ₹{july_v3_rate:,.2f}")
print(f"Observed improvement: {v3_observed_incremental_rate:,.2f} per account")
print(f"Observed July incremental recovery from v3: ₹{v3_july_incremental_recovery:,.2f}")

June v3 recovery/targeted account: ₹5,515.67
July v3 recovery/targeted account: ₹7,483.78
Observed improvement: 1,968.11 per account
Observed July incremental recovery from v3: ₹3,113,543.19


In [18]:
## 5. Golden Dataset Rules & Assumptions

The Golden Dataset is designed to provide a reproducible analytical layer from the raw operational data.

### Source-of-truth decisions

- `payment_id` is used as the canonical payment-event identifier.
- `account_id` is used as the canonical account identifier.
- `agent_id` is used as the operational agent identifier across calls, attempts, and sessions.
- Campaign attributes are taken from the campaign master and linked through `campaign_id`.

### Deduplication

- Exact duplicate payment records are removed.
- Repeated `payment_id` records are reduced to one canonical payment event.
- `payment_reference` is not used as the primary deduplication key because the same reference can map to multiple payment IDs, accounts, and amounts.

### Timestamp treatment

- Event timestamps are interpreted using the source timezone where available.
- Timestamps are normalized to UTC before deriving analytical date/hour fields.
- Events outside the analysis window are retained in raw data but excluded from period-based performance metrics unless explicitly required.

### Missing data

- Missing identifiers are not artificially imputed.
- Missing `payment_reference` does not cause a payment to be removed when a valid `payment_id` exists.
- Unmapped entities are retained with an explicit unknown/unmapped classification where required for analysis.

### Payment treatment

- Only `payment_status = SUCCESS` contributes to successful recovery.
- Recovery is based on the deduplicated payment-event layer.
- Payment attribution to campaigns is based on the defined analytical attribution window and is treated as an assumption where historical attribution cannot be established exactly.

### Exclusions

- Exact duplicate payment records are excluded from the Golden Dataset.
- Partial-month August data is excluded from June-to-July MoM comparison.
- Records failing essential identifier or timestamp validation are flagged and excluded from the relevant metric rather than silently discarded.

### Key assumptions

- `payment_id` represents the most reliable transaction-level identifier available.
- The supplied source timezone represents the local timezone of the recorded event.
- Same-month targeting-to-payment attribution is used for the campaign analysis performed in this assignment.

These rules are documented so that the analytical results can be reproduced and audited.

SyntaxError: invalid syntax (1900618003.py, line 3)

In [19]:
from pathlib import Path
import pandas as pd

BASE_PATH = Path("/content/collections_30k_dataset")

payments = pd.read_csv(BASE_PATH / "payments.csv")

print("Payments loaded:", len(payments))
print(payments.columns.tolist())

Payments loaded: 25500
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']


In [20]:
# Golden payment dataset

gold_payments = (
    payments
    .copy()
    .drop_duplicates()
)

gold_payments["event_at"] = pd.to_datetime(
    gold_payments["event_at"],
    errors="coerce"
)

# One canonical record per payment_id
gold_payments = (
    gold_payments
    .sort_values(["payment_id", "event_at"])
    .drop_duplicates("payment_id", keep="first")
    .reset_index(drop=True)
)

print("Raw payment rows:", len(payments))
print("Golden payment rows:", len(gold_payments))
print("Rows removed:", len(payments) - len(gold_payments))

Raw payment rows: 25500
Golden payment rows: 25000
Rows removed: 500


In [21]:
print(
    "Duplicate payment IDs remaining:",
    gold_payments["payment_id"].duplicated().sum()
)

print(
    "Duplicate complete rows remaining:",
    gold_payments.duplicated().sum()
)

print(
    "Missing payment_id:",
    gold_payments["payment_id"].isna().sum()
)

Duplicate payment IDs remaining: 0
Duplicate complete rows remaining: 0
Missing payment_id: 0


In [22]:
# Export Golden Payments Dataset
gold_payments.to_csv(
    "/content/gold_payments.csv",
    index=False
)

print("Saved:", "/content/gold_payments.csv")

Saved: /content/gold_payments.csv


In [26]:
from pathlib import Path
import pandas as pd

BASE_PATH = Path("/content/collections_30k_dataset")

payments = pd.read_csv(BASE_PATH / "payments.csv")
daily_targeting = pd.read_csv(BASE_PATH / "daily_targeting.csv")
campaigns = pd.read_csv(BASE_PATH / "campaigns.csv")

In [27]:
payments["event_at"] = pd.to_datetime(
    payments["event_at"],
    errors="coerce"
)

payments["month"] = payments["event_at"].dt.to_period("M")

successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

raw_monthly = (
    successful_payments
    .groupby("month")["amount"]
    .sum()
    .rename("raw_recovery")
)

dedup_successful = (
    successful_payments
    .drop_duplicates("payment_id")
)

dedup_monthly = (
    dedup_successful
    .groupby("month")["amount"]
    .sum()
    .rename("dedup_recovery")
)

daily_targeting["target_date"] = pd.to_datetime(
    daily_targeting["target_date"],
    errors="coerce"
)

daily_targeting["month"] = (
    daily_targeting["target_date"].dt.to_period("M")
)

monthly_targeting = (
    daily_targeting
    .groupby("month")["account_id"]
    .nunique()
    .rename("targeted_accounts")
)

recovery_comparison = pd.concat(
    [raw_monthly, dedup_monthly, monthly_targeting],
    axis=1
).reset_index()

recovery_comparison["raw_recovery_per_targeted"] = (
    recovery_comparison["raw_recovery"]
    / recovery_comparison["targeted_accounts"]
)

recovery_comparison["dedup_recovery_per_targeted"] = (
    recovery_comparison["dedup_recovery"]
    / recovery_comparison["targeted_accounts"]
)

print("Monthly recovery table created.")
display(recovery_comparison)

Monthly recovery table created.


,month,raw_recovery,dedup_recovery,targeted_accounts,raw_recovery_per_targeted,dedup_recovery_per_targeted
0,2026-01,"191,133,284.42","187,229,127.59",5732,"33,344.96","32,663.84"
1,2026-02,"174,097,287.96","170,142,453.76",5160,"33,739.78","32,973.34"
2,2026-03,"193,233,384.29","188,912,374.02",5666,"34,104.02","33,341.40"
3,2026-04,"178,427,017.05","175,138,043.41",5585,"31,947.54","31,358.65"
4,2026-05,"187,048,144.34","184,250,278.50",5800,"32,249.68","31,767.29"
5,2026-06,"178,724,493.46","175,559,726.97",5535,"32,289.88","31,718.11"
6,2026-07,"190,278,846.88","187,242,265.08",5666,"33,582.57","33,046.64"
7,2026-08,"48,543,467.93","47,109,695.31",1566,"30,998.38","30,082.82"


In [28]:
targeting = daily_targeting.copy()

targeting["month"] = (
    targeting["target_date"].dt.to_period("M")
)

strategy_targets = (
    targeting
    .groupby(["month", "campaign_id"])
    .agg(
        targeted_accounts=("account_id", "nunique")
    )
    .reset_index()
)

strategy_targets = strategy_targets.merge(
    campaigns[
        ["campaign_id", "strategy_version"]
    ],
    on="campaign_id",
    how="left"
)

# Successful payments
payment_strategy = (
    successful_payments[
        ["account_id", "amount", "month"]
    ]
    .merge(
        targeting[
            ["account_id", "campaign_id", "target_date"]
        ],
        on="account_id",
        how="inner"
    )
)

payment_strategy["target_month"] = (
    payment_strategy["target_date"].dt.to_period("M")
)

payment_strategy = payment_strategy[
    payment_strategy["month"] == payment_strategy["target_month"]
]

strategy_recovery = (
    payment_strategy
    .groupby(["month", "campaign_id"])["amount"]
    .sum()
    .rename("recovery")
    .reset_index()
)

strategy_performance = (
    strategy_targets
    .merge(
        strategy_recovery,
        on=["month", "campaign_id"],
        how="left"
    )
)

strategy_performance["recovery"] = (
    strategy_performance["recovery"].fillna(0)
)

strategy_performance["recovery_per_targeted"] = (
    strategy_performance["recovery"]
    / strategy_performance["targeted_accounts"]
)

strategy_performance = (
    strategy_performance
    .groupby(["month", "strategy_version"])
    .agg(
        targeted_accounts=("targeted_accounts", "sum"),
        recovery=("recovery", "sum")
    )
    .reset_index()
)

strategy_performance["recovery_per_targeted"] = (
    strategy_performance["recovery"]
    / strategy_performance["targeted_accounts"]
)

print("Strategy performance table created.")
display(strategy_performance)

Strategy performance table created.


,month,strategy_version,targeted_accounts,recovery,recovery_per_targeted
0,2026-01,legacy,1999,"14,094,676.17","7,050.86"
1,2026-01,v1,1295,"6,990,077.57","5,397.74"
2,2026-01,v2,1378,"8,260,356.59","5,994.45"
3,2026-01,v3,1687,"12,004,553.22","7,115.92"
4,2026-02,legacy,1743,"9,623,971.34","5,521.50"
5,2026-02,v1,1148,"6,435,254.00","5,605.62"
6,2026-02,v2,1243,"6,962,734.26","5,601.56"
7,2026-02,v3,1569,"9,285,843.17","5,918.32"
8,2026-03,legacy,1917,"14,484,230.14","7,555.68"
9,2026-03,v1,1259,"9,098,016.83","7,226.38"


In [29]:
campaign_channel = campaigns[
    ["campaign_id", "channel"]
]

channel_targets = targeting.merge(
    campaign_channel,
    on="campaign_id",
    how="left"
)

channel_targets_summary = (
    channel_targets
    .groupby(["month", "channel"])["account_id"]
    .nunique()
    .rename("targeted_accounts")
    .reset_index()
)

channel_recovery = (
    payment_strategy
    .merge(
        campaign_channel,
        on="campaign_id",
        how="left"
    )
    .groupby(["month", "channel"])["amount"]
    .sum()
    .rename("recovery")
    .reset_index()
)

channel_performance = (
    channel_targets_summary
    .merge(
        channel_recovery,
        on=["month", "channel"],
        how="left"
    )
)

channel_performance["recovery"] = (
    channel_performance["recovery"].fillna(0)
)

channel_performance["recovery_per_targeted"] = (
    channel_performance["recovery"]
    / channel_performance["targeted_accounts"]
)

print("Channel performance table created.")
display(channel_performance)

Channel performance table created.


,month,channel,targeted_accounts,recovery,recovery_per_targeted
0,2026-01,FIELD,987,"7,090,486.14","7,183.88"
1,2026-01,MIXED,1189,"7,007,830.10","5,893.89"
2,2026-01,SMS,1458,"9,831,646.32","6,743.24"
3,2026-01,VOICE,1017,"7,193,733.30","7,073.48"
4,2026-01,WHATSAPP,1567,"10,225,967.69","6,525.82"
5,2026-02,FIELD,814,"4,239,535.89","5,208.28"
6,2026-02,MIXED,1117,"8,055,937.22","7,212.12"
7,2026-02,SMS,1295,"7,152,762.93","5,523.37"
8,2026-02,VOICE,946,"4,899,597.49","5,179.28"
9,2026-02,WHATSAPP,1411,"7,959,969.24","5,641.37"


In [30]:
gold_payments.to_csv(
    "/content/gold_payments.csv",
    index=False
)

recovery_comparison.to_csv(
    "/content/monthly_recovery.csv",
    index=False
)

strategy_performance.to_csv(
    "/content/strategy_performance.csv",
    index=False
)

channel_performance.to_csv(
    "/content/channel_performance.csv",
    index=False
)

print("All four Power BI CSVs created.")

All four Power BI CSVs created.
